# Lab A — 用 `adk eval` 評保健品 agent（Colab · Service Account 版）

延續你蓋的**保健品文案小組**，這一關**回頭評它**：走對流程沒（trajectory）＋ 產出夠好沒（response）。

> **認證用 Service Account JSON**（不是 `auth.authenticate_user()`）：公司政策擋「第三方 notebook 用**你的 Google 帳號**存取 GCP」，
> 但 **SA 是「機器人身分」、不走你的帳號 OAuth → 繞過封鎖**，走**真的 Vertex**（Lab B 也共用同一份）。**Runtime → Run all**。

## 0. 安裝（google-adk 的 `[eval]` 提供 adk eval）

In [ ]:
!pip install -q "google-adk[eval]==1.37.0"

## 1. 上傳 Service Account JSON（講師提供）

執行這格 → 跳出上傳鈕 → 選講師給的 `*.json`。
> ⚠️ 這是憑證，別外流、別 commit 進 git；課後講師會停用該 SA。

In [ ]:
from google.colab import files
import os, json

print("請上傳 Service Account JSON 檔案（講師提供）：")
uploaded = files.upload()
sa_filename = os.path.abspath(list(uploaded.keys())[0])
with open(sa_filename) as f:
    sa_info = json.load(f)

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = sa_filename
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["GOOGLE_CLOUD_PROJECT"] = sa_info["project_id"]
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"
print("✅ Vertex 認證完成，project =", sa_info["project_id"])

## 2. 拿 agent 程式碼（clone repo）＋寫 .env

In [ ]:
!git clone -q https://github.com/amelielee-tech/adk-workshop.git

with open("adk-workshop/.env", "w") as f:
    f.write(
        f"GOOGLE_APPLICATION_CREDENTIALS={os.environ['GOOGLE_APPLICATION_CREDENTIALS']}\n"
        f"GOOGLE_GENAI_USE_VERTEXAI=TRUE\n"
        f"GOOGLE_CLOUD_PROJECT={os.environ['GOOGLE_CLOUD_PROJECT']}\n"
        f"GOOGLE_CLOUD_LOCATION=global\n"
    )
print("repo cloned；.env 寫好（Vertex + SA）")

## 2.5 先看清楚「我們拿什麼在評」（考卷＋標準答案＋及格線）

- `copy_agent.evalset.json`＝**考卷＋標準答案**（問句＋期望軌跡＋期望回覆）
- `test_config.json`＝**及格線**（哪些指標、門檻多少）
- `lab2_multi_agent/`＝**被評的對象**（你上一堂蓋的 agent）

In [ ]:
import json

print("── lab_eval/ 裡的評估檔案 ──")
!ls -1 adk-workshop/lab_eval/
print()

print("=" * 56)
print("test_config.json（及格線 / grader）")
print("=" * 56)
print(open("adk-workshop/lab_eval/test_config.json").read())

print("=" * 56)
print("copy_agent.evalset.json（考卷 / 測試案例）")
print("=" * 56)
ev = json.load(open("adk-workshop/lab_eval/copy_agent.evalset.json"))
print(f"eval_set_id: {ev['eval_set_id']}｜案例數: {len(ev['eval_cases'])}\n")
for case in ev["eval_cases"]:
    inv = case["conversation"][0]
    print("● 案例:", case["eval_id"])
    print("   ① user 問句　　　：", inv["user_content"]["parts"][0]["text"])
    print("   ② 期望軌跡(trajectory)：",
          [t["name"] + str(t.get("args", {})) for t in inv["intermediate_data"]["tool_uses"]])
    print("   ③ 期望回覆(response)：", inv["final_response"]["parts"][0]["text"][:48], "...\n")

print(r"""
被評的 agent（lab2_multi_agent）流程：

campaign_coordinator（root：把需求交給 pipeline）
        |
        v
campaign_pipeline（SequentialAgent，依序跑）
  (1) trend_researcher     -> get_market_trends    （市場趨勢）
  (2) audience_researcher  -> get_audience_profile （客群輪廓）
  (3) write_review_loop（LoopAgent，最多 3 輪）
         writer 寫稿 -> reviewer 用 check_copy_format 檢查
             |-- 不過 -> record_revision -> 退回 writer 重寫
             \\-- 過   -> approve_copy    -> 跳出迴圈
  (4) finalizer            -> 定稿輸出

對照你等下在軌跡看到的工具：transfer_to_agent / get_market_trends /
get_audience_profile / check_copy_format / approve_copy —— 就是這條線走出來的。
""")

## 3. 跑 `adk eval`（評 lab2_multi_agent）

`adk eval` **〈評誰〉〈用哪份考卷〉** `--config`**〈及格線〉**。會真的把 agent 跑兩次，約 1–2 分鐘。

In [ ]:
!cd adk-workshop && adk eval lab2_multi_agent lab_eval/copy_agent.evalset.json \
  --config_file_path lab_eval/test_config.json 2>&1 | grep -E "Tests passed|Tests failed|Using evaluation" 

## 4. 讀成乾淨的分數表（兩軸分開看）

In [ ]:
import json, glob
import pandas as pd

rows = []
for f in sorted(glob.glob("adk-workshop/lab2_multi_agent/.adk/eval_history/*.json")):
    d = json.load(open(f))
    for c in d["eval_case_results"]:
        r = {"case": c["eval_id"]}
        for m in c["overall_eval_metric_results"]:
            r[m["metric_name"]] = round(m["score"], 3)
            r[m["metric_name"] + " → "] = "PASS" if m["eval_status"] == 1 else "FAIL"
        rows.append(r)

pd.DataFrame(rows)

**怎麼讀**：
- `tool_trajectory_avg_score`＝走對流程沒。門檻 1.0、用 IN_ORDER。
- `response_match_score`＝文案字面（ROUGE-1 F1）像不像。**中文創作型天生低又飄**（門檻只 0.15）——這正是下一關 Lab B 改用 LLM-judge 的理由。

## 5. 效率：打一次 agent 看 latency + token

In [ ]:
!cd adk-workshop && python lab_eval/probe_efficiency.py

## 收尾

- **agent 評估分兩軸**：走對流程（trajectory）＋ 產出夠好（response）。
- **每個指標都有極限**：ROUGE 對創作型中文很弱 → 所以 Lab B 用 LLM-judge。
- **效率**（latency/token）只有真的跑 agent 才量得到。

一句話：**eval 讓「感覺不錯」變成「量得出來」。**